In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import xarray as xr
import numpy as np
import os
from glob import glob
from mpl_toolkits.basemap import Basemap
from numpy import meshgrid
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import cartopy.feature as cfeature
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LatitudeLocator
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable
from matplotlib import colormaps
import pandas as pd
import math
from datetime import datetime
import datetime as dt
from ridgeplot import ridgeplot
import joypy
import seaborn as sns
from matplotlib import cm
import climpred
from xclim import sdba
from climpred.options import OPTIONS
import json
from sklearn.metrics import roc_curve, auc, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from matplotlib.lines import Line2D  # For custom legend entries
import warnings
from sklearn.exceptions import UndefinedMetricWarning
import matplotlib.gridspec as gridspec
import hydroeval as he
import re
from matplotlib.colors import Normalize
import matplotlib.colors as mcolors
import dill

from function import preprocessUtils as putils
from function import masks
from function import verifications
from function import funs as f
from function import conf
from function import loadbias
from function import quikplot as qp
from function import dataLoad
from function import conf


warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

global dim_order, region_name, test_year, leads_
dim_order = conf.dim_order

test_year = 2019
leads_ = [6,13,20,27]

dir = '/glade/work/klesinger/FD_RZSM_deep_learning'
assert test_year == 2019, 'This is only the script for when the testing years are 2018-2019. Test year must = 2019.'


NameError: name 't' is not defined

## For this script, we are going to create a CRPS based on climatology.

In [ ]:
region_name = 'CONUS' #['australia','china','CONUS']
obs_source = 'GLEAM' #['GLEAM','ERA5']
week_lead=3
day_num = (week_lead*7)-1

if obs_source == 'ERA5':
    soil_dir = conf.era_data
elif obs_source == 'GLEAM':
    soil_dir = conf.gleam_data

In [ ]:
mask, mask_anom = masks.load_mask_vals(region_name)
try:
    mask = mask.rename({'X':'longitude','Y':'latitude'})
except ValueError:
    pass

In [ ]:
global custom_names
'''This is for the final plot for ACC and CRPS'''
custom_names = {
    'GEFSv12': 'GEFSv12','GEFSv12-BC': 'GEFSv12-BC', 'DL-DM_GEFSv12': 'DL-DM-GEFSv12','DL-DM_ECMWF': 'DL-DM-ECMWF',
    'ECMWF':'ECMWF', 'ECMWF-BC':'ECMWF-BC',
}

        
def return_name(name):
    if 'XGBOOST' in name:
        name_out = 'ML_NWP_OBS'
    else:
        name_out = name
    custom_names = {name: name_out}

    return(custom_names)

In [ ]:
'''Testing and validation dates only for the year 2019'''

test_start = '2018-01-01'
test_end = '2019-12-31'
val_start = '2016-01-01'
val_end = '2017-12-31'
train_start  = '2000-01-01'
train_end = '2015-12-31'

In [ ]:
global obs_anomaly,obs_raw
obs_anomaly,obs_raw = dataLoad.load_rzsm_observations(soil_dir, region_name)
obs_anomaly["time"] = obs_anomaly["time"].dt.floor("D")
obs_raw["time"] = obs_raw["time"].dt.floor("D")

try:
    obs_anomaly = obs_anomaly.rename({'X':'longitude','Y':'latitude'})
except ValueError:
    pass

obs_anomaly_to_sample_training = obs_anomaly.sel(time=slice(train_start,train_end))
obs_anomaly_to_sample_training['week_of_year'] = (
    'time', 
    obs_anomaly_to_sample_training['time'].dt.weekofyear.data
)
obs_anomaly_to_sample_training['year'] = (
    'time', 
    obs_anomaly_to_sample_training['time'].dt.year.data
)
obs_anomaly_to_sample_training

In [ ]:
'''Test subsets of obs, ecmwf raw, gefsv12 raw '''
global obs_anomaly_SubX_format, baseline_gefs, baseline_ecmwf, var_OUT, template_testing_only
obs_anomaly_SubX_format, baseline_gefs, baseline_ecmwf, var_OUT, template_testing_only = verifications.open_obs_and_baseline_files_multiple_leads(region_name, leads_, train_start, test_end, mask_anom,soil_dir)

init_dates, dt_dates, only_testing_dates = dataLoad.return_init_and_testing_dates(region_name,test_start,test_end)

In [ ]:
#Now that we have the observation file to overwrite, we want to randomly sample from the observations for that week of the year
obs_training = obs_anomaly_SubX_format.sel(L=day_num).sel(S=slice(train_start,train_end))
obs_training

In [ ]:
obs_testing = obs_anomaly_SubX_format.sel(L=day_num).sel(S=slice(test_start,test_end))
obs_testing


In [ ]:
def save_climatology_by_lead(week_lead,region_name, day_num, obs_testing,obs_anomaly_to_sample_training,obs_source):

    save_dir = f'Data/climatology_sample/{region_name}'
    os.makedirs(save_dir,exist_ok=True)

    save_file = f'{save_dir}/Wk{week_lead}_climatology_{obs_source}.nc'

    if not os.path.exists(save_file):
    
        #Now overwrite files with the Week of year randomly sampled from observations
        
        # Step 1: add 20 days to each forecast init time (S)
        forecast_weeks = (obs_testing['S'] + np.timedelta64(day_num, 'D')).dt.isocalendar().week.values
        
        # Step 2: Prepare new forecast array
        new_forecast = obs_testing.copy()
        
        obs = obs_anomaly_to_sample_training.copy()
        
        
        for s_idx, init_time in enumerate(obs_testing['S'].values):
            print(f'Working on init {s_idx}')
            target_week = forecast_weeks[s_idx]
        
            for y_idx, lat in enumerate(obs_testing['Y'].values):
                for x_idx, lon in enumerate(obs_testing['X'].values):
                    
                    # Find obs values matching this week at this grid point
                    matching_obs = obs.sel(
                        time=obs['week_of_year'] == target_week,
                        latitude=lat,
                        longitude=lon,
                        method='nearest'
                    )
        
                    # Match obs at nearest lat/lon
                    obs_point = obs.sel(latitude=lat, longitude=lon, method='nearest')
        
                    # Filter for target week
                    week_data = obs_point.where(obs_point['week_of_year'] == target_week, drop=True)
        
                    # Group by year and randomly pick one per year
                    df = week_data.to_dataframe().reset_index().dropna(subset=['RZSM'])
        
                    # Get one sample per year
                    df_yearly = df.groupby('year').apply(lambda x: x.sample(1)).reset_index(drop=True)
        
                    # Randomly sample 11 distinct years if enough available
                    if len(df_yearly) >= 11:
                        selected_values = df_yearly.sample(11, replace=False)['RZSM'].values
                    elif len(df_yearly) > 0:
                        selected_values = df_yearly.sample(11, replace=True)['RZSM'].values
                    else:
                        selected_values = np.full((11,), np.nan)
        
                    # Assign to forecast
                    new_forecast['var'][s_idx, :, y_idx, x_idx] = selected_values
                    
        new_forecast.to_netcdf(save_file)
        return new_forecast
    else:
        return(xr.open_dataset(save_file))
        


In [ ]:
obs_clim = save_climatology_by_lead(week_lead,region_name, day_num, obs_testing,obs_anomaly_to_sample_training,obs_source)
obs_clim = obs_clim.expand_dims({'L':1}).rename({'var':'RZSM'})
obs_clim

In [ ]:
#Compute the CRPS
global obs_original,obs_raw

obs_original,obs_raw = dataLoad.load_rzsm_observations(soil_dir, region_name)
'''climpred automatically changes to pentads, but I need weeks'''


# obs_clim['L'] = np.ceil(obs_clim['L'].values/7)
xr.set_options(keep_attrs=True)
obs_clim['L'].attrs['units'] = 'not_a_time'
obs_clim['L'].attrs

'''Must change the day to ensure climpred computes correctly and changes attrs to weeks'''
obs_clim['L']=obs_clim['L']+1

obs_CRPS = verifications.create_climpred_CRPS(verifications.rename_subx_for_climpred(obs_clim), verifications.rename_obs_for_climpred(obs_original))
obs_CRPS['lead'].attrs

save_dir = f'Data/climatology_crps/{region_name}'
os.makedirs(save_dir,exist_ok=True)

obs_CRPS.to_netcdf(f'{save_dir}/Wk{week_lead}_climatology_{obs_source}.nc')


In [ ]:
obs_CRPS.mean()

In [ ]:
obs_CRPS['lead'].attrs

In [ ]:
obs_CRPS

# Load the EXperiment CRPS and plot